In [1]:
import pyspark

In [2]:
from pyspark.sql.functions import *

In [3]:
from pyspark.sql import SparkSession

In [4]:
# download the data
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet

--2026-05-12 23:08:11--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.190, 3.171.57.103, 3.171.57.179, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49961641 (48M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-01.parquet’

yellow_tripdata_202 100%[===================>]  47.65M   104MB/s    in 0.5s    

2026-05-12 23:08:12 (104 MB/s) - ‘yellow_tripdata_2024-01.parquet’ saved [49961641/49961641]



In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-05-12 23:09:05--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.190, 3.171.57.179, 3.171.57.69, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-05-12 23:09:05 (1.17 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [6]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 23:09:29 WARN Utils: Your hostname, codespaces-ba9731, resolves to a loopback address: 127.0.0.1; using 10.0.10.49 instead (on interface eth0)
26/05/12 23:09:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 23:09:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

In [9]:
total_trips = df.count()
total_trips

2964624

In [10]:
print(df.columns)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


In [11]:
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [12]:
missing_values = df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
)
missing_values.show()

[Stage 8:=============================>                             (1 + 1) / 2]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         140162|            0|    140162|            140162|           0|           0|           0|          0|    0|      0|         

In [14]:
trip_distance_stats = df.select(
    min("trip_distance").alias("min_trip_distance"),
    max("trip_distance").alias("max_trip_distance"),
    avg("trip_distance").alias("avg_trip_distance"),
)
trip_distance_stats.show()

+-----------------+-----------------+------------------+
|min_trip_distance|max_trip_distance| avg_trip_distance|
+-----------------+-----------------+------------------+
|              0.0|         312722.3|3.6521691789580624|
+-----------------+-----------------+------------------+



In [15]:
# How long does the average taxi trip last?
df_with_time = df.withColumn(
    "pickup_hour", hour(col("tpep_pickup_datetime"))
).withColumn(
    "duration_minutes",
    round(
        (
            unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime")
        )
        / 60,
        2,
    ),
)

valid_duration_df = df_with_time.filter(col("duration_minutes") > 0)

average_duration = valid_duration_df.agg(
    round(avg("duration_minutes"), 2).alias("avg_trip_duration_minutes")
)

average_duration.show()

[Stage 14:=============================>                            (1 + 1) / 2]

+-------------------------+
|avg_trip_duration_minutes|
+-------------------------+
|                    15.62|
+-------------------------+



In [16]:
zones_df = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

In [17]:
zones_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [20]:
zones_df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [22]:
# What are the top pickup zones by trip count?
pickup_zones = zones_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone"),
    col("service_zone").alias("pickup_service_zone"),
)

In [23]:
taxi_with_pickup_zone = df.join(pickup_zones, on="PULocationID", how="left")


In [24]:
top_pickup_zones = (
    taxi_with_pickup_zone.groupBy("pickup_borough", "pickup_zone")
    .agg(count("*").alias("trip_count"))
    .orderBy(desc("trip_count"))
)

top_pickup_zones.show(10, truncate=False)

[Stage 22:=============================>                            (1 + 1) / 2]

+--------------+----------------------------+----------+
|pickup_borough|pickup_zone                 |trip_count|
+--------------+----------------------------+----------+
|Queens        |JFK Airport                 |145240    |
|Manhattan     |Midtown Center              |143471    |
|Manhattan     |Upper East Side South       |142708    |
|Manhattan     |Upper East Side North       |136465    |
|Manhattan     |Midtown East                |106717    |
|Manhattan     |Times Sq/Theatre District   |106324    |
|Manhattan     |Penn Station/Madison Sq West|104523    |
|Manhattan     |Lincoln Square East         |104080    |
|Queens        |LaGuardia Airport           |89533     |
|Manhattan     |Upper West Side South       |88474     |
+--------------+----------------------------+----------+
only showing top 10 rows


In [25]:
top_pickup_zones.write.options(header='True', delimiter=',').csv("top_pickup_zones")